#Transform Results Data
1. Read bronze results table
2. Keep only the columns required for analytics (Drop url column)
3. Standardize column names using snake_case (driverId → driver_id, constructorId → constructor_id,raceName → race_name,positiontext → finish_position_text )
4.Rename columns to make them more meaningful (date → race_date, grid → grid_position, laps → completed_laps, number → car_number, position → finish_position 
5. Filter out rows where season, round, constructor_id and driver_id is null (business key validation)
6. Remove duplicate records
7. Transform column values of race_name to Title Case
8. Write the transformed data to silver results table

In [0]:
%run ../00-common/01.environment_config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.results"
silver_table = f"{catalog_name}.{silver_schema}.results"


Step 1-Read bronze results table

In [0]:
results_df = spark.table(bronze_table)
#                            (OR)
# results_df = spark.read.option("versionAsOf",0).table(bronze_table)

In [0]:
display(results_df)

Step 2- Keep only the columns required for analytics (Drop url column)

In [0]:
from pyspark.sql import functions as F
"""
results_selected_df = results_df.select(
                    "date",
                    "raceName",
                    "round",
                    "season"
                    "constructorId"
                    "driverId"
                    "grid"
                    "laps"
                    "number"
                    "points"
                    "position"
                    "positionText"
                    "status"
                    "ingestion_timestamp"
                    "source_file")
)"""

results_selected_df = results_df.drop("url")


In [0]:
display(results_selected_df)

Step 3 & 4:
- Standardize column names using snake_case (driverId → driver_id, constructorId → constructor_id,raceName → race_name,positiontext → finish_position_text )
- Rename columns to make them more meaningful (date → race_date, grid → grid_position, laps → completed_laps, number → car_number, position → finish_position

In [0]:
"""circuits_renamed_df = (

        circuits_selected_df
            .withColumnRenamed("driverId","driver_id")
            .withColumnRenamed("raceName","race_name")
            .withColumnRenamed("constructorId","constructor_id") 
            .withColumnRenamed("positionText","finsih_position_text") 
            .withColumnRenamed("position","finsih_position") 
            .withColumnRenamed("grid","grid_position") 
            .withColumnRenamed("laps","completed_laps") 
            .withColumnRenamed("number","car_number") 
            .withColumnRenamed("date","race_date")
            
                                                    )

                      OR                             """
results_renamed_df = (results_selected_df
                       .withColumnsRenamed({
                           "driverId":"driver_id",
                           "constructorId":"constructor_id",
                           "positionText":"finsih_position_text",
                           "raceName":"race_name",
                           "date": "race_date",
                           "grid":"grid_position",
                           "laps":"completed_laps",
                           "number":"car_number",
                           "position":"finish_position"})
                       )


display(results_renamed_df)




Step 5- Filter out rows where season, round, constructor_id and driver_id is null (business key validation)

In [0]:
results_valid_df = results_renamed_df.filter(
                                            F.col("season").isNotNull() & 
                                            F.col("round").isNotNull() & 
                                            F.col("constructor_id").isNotNull() & 
                                            F.col("driver_id").isNotNull()
)
results_renamed_df.count()-results_valid_df.count()

Step 6- Remove duplicate records

In [0]:
"""results_distinct_df = results_renamed_df.distinct()
                        OR  """
results_distinct_df = results_valid_df.dropDuplicates(["driver_id","season","round","constructor_id"])
results_valid_df.count()-results_distinct_df.count()

Step 7- Transform column values of race_name to Title Case

In [0]:
results_final_df = (results_distinct_df
                            .withColumn("race_name",F.initcap(F.col("race_name")))

)
results_final_df.display()



Step 8- Write the transformed data to silver results table

In [0]:
(
    results_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))